# Task 1: ET Error Visualization (High-Res 300+ DPI)

Trích xuất và trực quan hóa các ca lỗi tiêu biểu của vùng Enhancing Tumor (ET):
1. **GT_ET=0 với Dương tính giả (False Positive)**: Màng não / xoang tĩnh mạch (`BraTS2021_01437`) & viền hoại tử NCR (`BraTS2021_01511`).
2. **U nhỏ bị bỏ sót (False Negative)**: `BraTS2021_00380` (bỏ sót >91% thể tích) & `BraTS2021_01520` (vi khối u 109 voxels).
3. **U lớn xâm lấn lõi hoại tử (NCR)**: `BraTS2021_01441` & `BraTS2021_01562` (Standard 3D UNet tràn vào NCR vs nnU-Net phân định chuẩn xác).

Xuất file PNG độ phân giải cao $\ge 300$ DPI với 6 panels, nhãn GT, UNet, nnU-Net và Error classification map.

In [ ]:
# 1. Cài đặt các thư viện phụ thuộc
!pip install -q monai nibabel scipy matplotlib tqdm
!nvidia-smi

In [ ]:
# 2. Giải nén nhanh các ca mục tiêu từ BraTS2021_Training_Data.tar
import os
import subprocess
from pathlib import Path

tar_files = list(Path("/kaggle/input").rglob("BraTS2021_Training_Data.tar"))
assert len(tar_files) > 0, f"Không tìm thấy BraTS2021_Training_Data.tar trong /kaggle/input"
main_tar = str(tar_files[0])
print(f"Tìm thấy file dữ liệu: {main_tar}")

dest_dir = Path("/kaggle/tmp/cases")
dest_dir.mkdir(parents=True, exist_ok=True)

target_pids = [
    "BraTS2021_01437", "BraTS2021_01511", "BraTS2021_00380",
    "BraTS2021_01520", "BraTS2021_01441", "BraTS2021_01562",
    "BraTS2021_01480", "BraTS2021_01433", "BraTS2021_01502", "BraTS2021_01179",
    "BraTS2021_00390", "BraTS2021_01545", "BraTS2021_01063", "BraTS2021_01158",
    "BraTS2021_01140", "BraTS2021_01150"
]

# Dùng tar wildcard để giải nén cực nhanh trong vài giây
wildcards = [f"*{pid}*" for pid in target_pids]
cmd = ["tar", "-xf", main_tar, "-C", str(dest_dir), "--wildcards"] + wildcards
print("Đang giải nén các ca mục tiêu...")
subprocess.run(cmd, check=True)
print("Giải nén thành công! Các thư mục hiện có:")
!ls /kaggle/tmp/cases

In [ ]:
# 3. Clone repo để lấy mã nguồn kiến trúc chuẩn hóa
!rm -rf /kaggle/tmp/repo && mkdir -p /kaggle/tmp
!git clone -b baselineTrain https://github.com/nam21-12-06/test.git /kaggle/tmp/repo || git clone -b baselineTrain https://github.com/chitruonghf711/brats2021-3d-unet-study.git /kaggle/tmp/repo

import sys
sys.path.insert(0, "/kaggle/tmp/repo")

In [ ]:
# 4. Xác định đường dẫn checkpoint của Standard 3D UNet và nnU-Net
from pathlib import Path

unet_ckpts = list(Path("/kaggle/input").rglob("*unet*baseline*/**/best_metric_model.pth"))
if not unet_ckpts:
    unet_ckpts = list(Path("/kaggle/input").rglob("best_metric_model.pth"))

print("Tất cả checkpoint tìm thấy:")
for c in Path("/kaggle/input").rglob("best_metric_model.pth"):
    print(f" - {c}")

# Gán đường dẫn rõ ràng từ kernel_sources
unet_ckpt = None
nnunet_ckpt = None
for p in Path("/kaggle/input").rglob("best_metric_model.pth"):
    ps = str(p)
    if "3d-unet" in ps or "unet_baseline" in ps:
        unet_ckpt = str(p)
    elif "nnunet" in ps:
        nnunet_ckpt = str(p)

print(f"\nĐã chọn:")
print(f"Standard 3D UNet: {unet_ckpt}")
print(f"nnU-Net (DynUNet): {nnunet_ckpt}")
assert unet_ckpt is not None, "Không tìm thấy UNet checkpoint!"
assert nnunet_ckpt is not None, "Không tìm thấy nnU-Net checkpoint!"

In [ ]:
# 5. Thực thi script generate_et_error_cases.py để sinh ảnh 300 DPI
out_dir = "/kaggle/working/outputs/et_errors"
os.makedirs(out_dir, exist_ok=True)

# Tìm thư mục chứa các ca bệnh thực tế sau khi giải nén
data_dir = "/kaggle/tmp/cases"
if os.path.exists("/kaggle/tmp/cases/BraTS2021_Training_Data"):
    data_dir = "/kaggle/tmp/cases/BraTS2021_Training_Data"

!python /kaggle/tmp/repo/scripts/generate_et_error_cases.py \
    --data-dir "{data_dir}" \
    --unet-ckpt "{unet_ckpt}" \
    --nnunet-ckpt "{nnunet_ckpt}" \
    --output-dir "{out_dir}" \
    --dpi 300

print("\nDanh sách các ảnh đã được tạo ra:")
!ls -lh /kaggle/working/outputs/et_errors

In [ ]:
# 6. Hiển thị trực quan ngay trong notebook
from IPython.display import Image, display
from pathlib import Path

pngs = sorted(Path("/kaggle/working/outputs/et_errors").glob("*.png"))
for p in pngs:
    print(f"=== {p.name} ===")
    display(Image(filename=str(p), width=900))